<a href="https://colab.research.google.com/github/miriamamin1213-ux/HPV-classification/blob/main/hpvstatus2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import torch
import torch.nn as nn
import torch.optim as optim

In [5]:
df = pd.read_excel("HPVstatus.xlsx")

In [6]:
df.head()

,PatientID,CenterID,Task 1,Task 2,Task 3,Age,Gender,Tobacco Consumption,Alcohol Consumption,Performance Status,Treatment,T-stage,N-stage,M-stage,HPV Status,Relapse,RFS
0,CHUM-001,1,1,1,0,82.0,1,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1704.0
1,CHUM-002,1,1,1,0,73.0,1,NaN,NaN,NaN,1.0,T3,N2,M0,NaN,1.0,439.0
2,CHUM-006,1,1,1,0,65.0,1,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1186.0
3,CHUM-007,1,1,1,0,70.0,0,NaN,NaN,NaN,0.0,T2,N2,M0,NaN,0.0,1702.0
4,CHUM-008,1,1,1,0,67.0,0,NaN,NaN,NaN,1.0,T2,N2,M0,NaN,0.0,1499.0


In [7]:
df.columns

Index(['PatientID', 'CenterID', 'Task 1', 'Task 2', 'Task 3', 'Age', 'Gender',
       'Tobacco Consumption', 'Alcohol Consumption', 'Performance Status',
       'Treatment', 'T-stage', 'N-stage', 'M-stage', 'HPV Status', 'Relapse',
       'RFS'],
      dtype='object')

In [8]:
df['HPV Status'].value_counts()

,count
HPV Status,
1.0,530
0.0,58


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 726 entries, 0 to 725
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   PatientID            726 non-null    object 
 1   CenterID             726 non-null    int64  
 2   Task 1               726 non-null    int64  
 3   Task 2               726 non-null    int64  
 4   Task 3               726 non-null    int64  
 5   Age                  726 non-null    float64
 6   Gender               726 non-null    int64  
 7   Tobacco Consumption  514 non-null    float64
 8   Alcohol Consumption  508 non-null    float64
 9   Performance Status   463 non-null    float64
 10  Treatment            707 non-null    float64
 11  T-stage              722 non-null    object 
 12  N-stage              724 non-null    object 
 13  M-stage              719 non-null    object 
 14  HPV Status           588 non-null    float64
 15  Relapse              678 non-null    flo

In [10]:
df[['T-stage','N-stage','M-stage']].head(10)

,T-stage,N-stage,M-stage
0,T2,N2,M0
1,T3,N2,M0
2,T2,N2,M0
3,T2,N2,M0
4,T2,N2,M0
5,T1,N2,M0
6,T3,N2,M0
7,T2,N1,M0
8,T1,N2,M0
9,T2,N1,M0


In [11]:
df['T-stage'].unique()

array(['T2', 'T3', 'T1', 'T4', nan, 'T0'], dtype=object)

In [12]:
df['N-stage'].unique()

array(['N2', 'N1', 'N0', 'N3', nan], dtype=object)

In [13]:
df['M-stage'].unique()

array(['M0', 'M1', nan], dtype=object)

In [14]:
df.isnull().sum()

,0
PatientID,0
CenterID,0
Task 1,0
Task 2,0
Task 3,0
Age,0
Gender,0
Tobacco Consumption,212
Alcohol Consumption,218
Performance Status,263


In [15]:
df['Treatment'].unique()

array([ 1.,  0., nan])

In [16]:
df[['Gender','Tobacco Consumption','Alcohol Consumption']].head(10)

,Gender,Tobacco Consumption,Alcohol Consumption
0,1,NaN,NaN
1,1,NaN,NaN
2,1,NaN,NaN
3,0,NaN,NaN
4,0,NaN,NaN
5,1,NaN,NaN
6,1,NaN,NaN
7,1,NaN,NaN
8,0,NaN,NaN
9,1,NaN,NaN


In [17]:
df['Gender'].value_counts(dropna=False)

,count
Gender,
1,610
0,116


In [18]:
df['Tobacco Consumption'].value_counts(dropna=False)

,count
Tobacco Consumption,
0.0,279
1.0,235
NaN,212


In [19]:
df['Alcohol Consumption'].value_counts(dropna=False)

,count
Alcohol Consumption,
1.0,324
NaN,218
0.0,184


In [20]:
df = df.dropna(subset=['HPV Status'])

In [21]:
print(df.shape)

(588, 17)


In [22]:
df['HPV Status'].value_counts()

,count
HPV Status,
1.0,530
0.0,58


In [23]:
df.isnull().sum()

,0
PatientID,0
CenterID,0
Task 1,0
Task 2,0
Task 3,0
Age,0
Gender,0
Tobacco Consumption,95
Alcohol Consumption,96
Performance Status,141


In [24]:
df[['Tobacco Consumption','Alcohol Consumption']].head(20)

,Tobacco Consumption,Alcohol Consumption
7,NaN,NaN
9,NaN,NaN
11,NaN,NaN
15,NaN,NaN
16,NaN,NaN
19,NaN,NaN
21,NaN,NaN
24,NaN,NaN
27,NaN,NaN
28,NaN,NaN


In [25]:
df[['Tobacco Consumption','Alcohol Consumption']].describe()

,Tobacco Consumption,Alcohol Consumption
count,493.000000,492.000000
mean,0.442191,0.636179
std,0.497151,0.481588
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,1.000000
75%,1.000000,1.000000
max,1.000000,1.000000


In [26]:
df[['Tobacco Consumption','Alcohol Consumption']].mode()

,Tobacco Consumption,Alcohol Consumption
0,0.0,1.0


In [27]:
tobacco_mode = df['Tobacco Consumption'].mode()[0]

In [28]:
df['Tobacco Consumption'] = df['Tobacco Consumption'].fillna(tobacco_mode)

In [29]:
alcohol_mode = df['Alcohol Consumption'].mode()[0]

df['Alcohol Consumption'] = df['Alcohol Consumption'].fillna(alcohol_mode)

In [30]:
df[['Tobacco Consumption','Alcohol Consumption']].isnull().sum()

,0
Tobacco Consumption,0
Alcohol Consumption,0


In [31]:
df.isnull().sum()

,0
PatientID,0
CenterID,0
Task 1,0
Task 2,0
Task 3,0
Age,0
Gender,0
Tobacco Consumption,0
Alcohol Consumption,0
Performance Status,141


In [32]:
df = df.drop(['Performance Status', 'Relapse', 'RFS'], axis=1)

In [33]:
df.columns

Index(['PatientID', 'CenterID', 'Task 1', 'Task 2', 'Task 3', 'Age', 'Gender',
       'Tobacco Consumption', 'Alcohol Consumption', 'Treatment', 'T-stage',
       'N-stage', 'M-stage', 'HPV Status'],
      dtype='object')

In [34]:
df = df.drop(['PatientID','CenterID','Task 1','Task 2','Task 3'], axis=1)

In [35]:
df.head()

,Age,Gender,Tobacco Consumption,Alcohol Consumption,Treatment,T-stage,N-stage,M-stage,HPV Status
7,61.0,1,0.0,1.0,1.0,T2,N1,M0,1.0
9,59.0,1,0.0,1.0,1.0,T2,N1,M0,1.0
11,58.0,1,0.0,1.0,1.0,T2,N2,M0,1.0
15,63.0,1,0.0,1.0,1.0,T4,N2,M0,1.0
16,56.0,1,0.0,1.0,1.0,T2,N2,M0,1.0


In [36]:
df.isnull().sum()

,0
Age,0
Gender,0
Tobacco Consumption,0
Alcohol Consumption,0
Treatment,19
T-stage,3
N-stage,2
M-stage,2
HPV Status,0


In [37]:
df = df.dropna()

In [38]:
df.shape

(562, 9)

In [39]:
df.isnull().sum()

,0
Age,0
Gender,0
Tobacco Consumption,0
Alcohol Consumption,0
Treatment,0
T-stage,0
N-stage,0
M-stage,0
HPV Status,0


In [40]:
df['T-stage'] = df['T-stage'].replace({
    'T0':0,
    'T1':1,
    'T2':2,
    'T3':3,
    'T4':4
})

/tmp/ipykernel_15985/630066328.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['T-stage'] = df['T-stage'].replace({


In [41]:
df['N-stage'] = df['N-stage'].replace({
    'N0':0,
    'N1':1,
    'N2':2,
    'N3':3,
})

/tmp/ipykernel_15985/3191464193.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['N-stage'] = df['N-stage'].replace({


In [42]:
df['M-stage'] = df['M-stage'].replace({
    'M0':0,
    'M1':1,
})

/tmp/ipykernel_15985/1125616347.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['M-stage'] = df['M-stage'].replace({


In [43]:
df.head()

,Age,Gender,Tobacco Consumption,Alcohol Consumption,Treatment,T-stage,N-stage,M-stage,HPV Status
7,61.0,1,0.0,1.0,1.0,2,1,0,1.0
9,59.0,1,0.0,1.0,1.0,2,1,0,1.0
11,58.0,1,0.0,1.0,1.0,2,2,0,1.0
15,63.0,1,0.0,1.0,1.0,4,2,0,1.0
16,56.0,1,0.0,1.0,1.0,2,2,0,1.0


In [44]:
X = df[['Age',
        'Gender',
        'Tobacco Consumption',
        'Alcohol Consumption',
        'Treatment',
        'T-stage',
        'N-stage',
        'M-stage']]

y = df['HPV Status']

In [45]:
print(X.shape)
print(y.shape)

(562, 8)
(562,)


In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [47]:
print(X_train.shape)
print(X_test.shape)

(449, 8)
(113, 8)


In [48]:
from sklearn.preprocessing import StandardScaler

In [49]:
scaler = StandardScaler()

In [50]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [51]:
print(X_train[:5])

[[-1.32266133  0.40399559 -0.7768986   0.66613055 -2.25112584 -0.18583296
  -2.6891386  -0.15092728]
 [ 0.23577337  0.40399559 -0.7768986   0.66613055  0.44422217 -0.18583296
  -1.12309031 -0.15092728]
 [ 1.34894101  0.40399559 -0.7768986   0.66613055  0.44422217 -1.14490194
   0.44295798 -0.15092728]
 [-0.54344398  0.40399559  1.28716927 -1.50120724  0.44422217  0.77323601
   0.44295798 -0.15092728]
 [ 1.90552483  0.40399559  1.28716927 -1.50120724  0.44422217  0.77323601
  -2.6891386  -0.15092728]]


In [52]:
type(X_train)

numpy.ndarray

In [53]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

In [54]:
print(type(X_train))
print(type(y_train))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [55]:
import torch.nn as nn

In [56]:
class HPVNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(8, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))

        return x

In [57]:
model = HPVNet()

In [58]:
print(model)

HPVNet(
  (fc1): Linear(in_features=8, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [59]:
model = HPVNet()
print(model)

HPVNet(
  (fc1): Linear(in_features=8, out_features=16, bias=True)
  (fc2): Linear(in_features=16, out_features=8, bias=True)
  (fc3): Linear(in_features=8, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [60]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [61]:
print(y_train.shape)

torch.Size([449])


In [62]:
y_train = y_train.view(-1, 1)
y_test = y_test.view(-1, 1)

In [63]:
print(y_train.shape)
print(y_test.shape)

torch.Size([449, 1])
torch.Size([113, 1])


In [64]:
epochs = 100

for epoch in range(epochs):

    # Forward pass
    outputs = model(X_train)

    # Calculate loss
    loss = criterion(outputs, y_train)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.8333
Epoch [20/100], Loss: 0.8207
Epoch [30/100], Loss: 0.8107
Epoch [40/100], Loss: 0.8014
Epoch [50/100], Loss: 0.7920
Epoch [60/100], Loss: 0.7814
Epoch [70/100], Loss: 0.7685
Epoch [80/100], Loss: 0.7517
Epoch [90/100], Loss: 0.7294
Epoch [100/100], Loss: 0.6993


In [65]:
with torch.no_grad():
    y_pred = model(X_test)

In [66]:
print(y_pred[:10])

tensor([[0.5366],
        [0.5210],
        [0.4680],
        [0.5106],
        [0.5096],
        [0.4693],
        [0.4920],
        [0.4786],
        [0.5557],
        [0.4992]])


In [67]:
predicted = (y_pred > 0.5).float()

print(predicted[:20])

tensor([[1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [0.],
        [0.],
        [1.],
        [0.],
        [1.],
        [0.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.]])


In [68]:
print(y_test.unique(return_counts=True))

(tensor([0., 1.]), tensor([14, 99]))


In [69]:
print(predicted.unique(return_counts=True))

(tensor([0., 1.]), tensor([65, 48]))


In [70]:
from imblearn.over_sampling import SMOTE

In [71]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train.numpy(),
    y_train.numpy().ravel()
)

In [72]:
import pandas as pd

print(pd.Series(y_train_smote).value_counts())

1.0    407
0.0    407
Name: count, dtype: int64


In [73]:
X_train_smote = torch.tensor(X_train_smote, dtype=torch.float32)

y_train_smote = torch.tensor(
    y_train_smote,
    dtype=torch.float32
).view(-1,1)

In [74]:
X_train_smote
y_train_smote

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
      

In [75]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train.numpy(),
    y_train.numpy().ravel()
)

In [76]:
import pandas as pd

print(pd.Series(y_train_smote).value_counts())

1.0    407
0.0    407
Name: count, dtype: int64


In [77]:
X_train_smote = torch.tensor(X_train_smote, dtype=torch.float32)

y_train_smote = torch.tensor(
    y_train_smote,
    dtype=torch.float32
).view(-1,1)

In [78]:
model = HPVNet()

In [79]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [80]:
X_train_smote
y_train_smote

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
      

In [81]:
predicted = (y_pred > 0.5).float()

In [82]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test.numpy(),
    predicted.numpy()
))

              precision    recall  f1-score   support

         0.0       0.12      0.57      0.20        14
         1.0       0.88      0.42      0.57        99

    accuracy                           0.44       113
   macro avg       0.50      0.50      0.39       113
weighted avg       0.78      0.44      0.53       113



In [83]:
print(predicted.unique(return_counts=True))

(tensor([0., 1.]), tensor([65, 48]))


In [84]:
model = HPVNet()

In [85]:
class HPVNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(8,32)
        self.fc2 = nn.Linear(32,16)
        self.fc3 = nn.Linear(16,1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))

        return x

In [86]:

model = HPVNet()

In [87]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [88]:
epochs = 100

In [89]:
model = HPVNet()
epochs = 100

In [90]:
class HPVNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(8,32)
        self.fc2 = nn.Linear(32,16)
        self.fc3 = nn.Linear(16,1)

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))

        return x

In [91]:
model = HPVNet()
print(model)

HPVNet(
  (fc1): Linear(in_features=8, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (fc3): Linear(in_features=16, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [92]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [93]:
X_train_smote
y_train_smote

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [1.],
      

In [94]:
with torch.no_grad():
    y_pred = model(X_test)

predicted = (y_pred > 0.5).float()

In [95]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test.numpy(),
    predicted.numpy()
))

              precision    recall  f1-score   support

         0.0       0.33      0.21      0.26        14
         1.0       0.89      0.94      0.92        99

    accuracy                           0.85       113
   macro avg       0.61      0.58      0.59       113
weighted avg       0.82      0.85      0.84       113



In [96]:
model = HPVNet()

In [97]:
criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [98]:
epochs = 200

In [99]:
for epoch in range(epochs):

    outputs = model(X_train_smote)

    loss = criterion(outputs, y_train_smote)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [20/200], Loss: 0.6716
Epoch [40/200], Loss: 0.6337
Epoch [60/200], Loss: 0.5741
Epoch [80/200], Loss: 0.5033
Epoch [100/200], Loss: 0.4462
Epoch [120/200], Loss: 0.4111
Epoch [140/200], Loss: 0.3894
Epoch [160/200], Loss: 0.3721
Epoch [180/200], Loss: 0.3571
Epoch [200/200], Loss: 0.3432


In [100]:
with torch.no_grad():
    y_pred = model(X_test)

predicted = (y_pred > 0.5).float()

In [101]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test.numpy(),
        predicted.numpy()
    )re code in one go please
recodeinonegoplease
)

              precision    recall  f1-score   support

         0.0       0.32      0.43      0.36        14
         1.0       0.91      0.87      0.89        99

    accuracy                           0.81       113
   macro avg       0.62      0.65      0.63       113
weighted avg       0.84      0.81      0.83       113



In [102]:
print(predicted.unique(return_counts=True))

(tensor([0., 1.]), tensor([19, 94]))


In [103]:
model = HPVNet()

In [104]:
criterion = nn.BCELoss()

In [105]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [106]:
epochs = 200

for epoch in range(epochs):

    outputs = model(X_train_smote)

    loss = criterion(outputs, y_train_smote)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [20/200], Loss: 0.7127
Epoch [40/200], Loss: 0.7083
Epoch [60/200], Loss: 0.7039
Epoch [80/200], Loss: 0.6996
Epoch [100/200], Loss: 0.6952
Epoch [120/200], Loss: 0.6907
Epoch [140/200], Loss: 0.6860
Epoch [160/200], Loss: 0.6811
Epoch [180/200], Loss: 0.6757
Epoch [200/200], Loss: 0.6700


In [107]:
with torch.no_grad():
    y_pred = model(X_test)

predicted = (y_pred > 0.5).float()

In [108]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test.numpy(),
        predicted.numpy()
    )
)

              precision    recall  f1-score   support

         0.0       0.12      1.00      0.22        14
         1.0       1.00      0.01      0.02        99

    accuracy                           0.13       113
   macro avg       0.56      0.51      0.12       113
weighted avg       0.89      0.13      0.05       113



In [110]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test.numpy(),
        predicted.numpy()
    )
)

              precision    recall  f1-score   support

         0.0       0.12      1.00      0.22        14
         1.0       1.00      0.01      0.02        99

    accuracy                           0.13       113
   macro avg       0.56      0.51      0.12       113
weighted avg       0.89      0.13      0.05       113



In [111]:
print(predicted.unique(return_counts=True))

(tensor([0., 1.]), tensor([112,   1]))


In [112]:
y_train_ce = torch.tensor(
    y_train_smote.numpy().flatten(),
    dtype=torch.long
)

y_test_ce = torch.tensor(
    y_test.numpy().flatten(),
    dtype=torch.long
)

In [113]:
class HPVNet2(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(8, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 2)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)

        return x

In [114]:
model = HPVNet2()

In [115]:
criterion = nn.CrossEntropyLoss()

In [116]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [117]:
epochs = 200

for epoch in range(epochs):

    outputs = model(X_train_smote)

    loss = criterion(outputs, y_train_ce)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(
            f'Epoch [{epoch+1}/{epochs}], '
            f'Loss: {loss.item():.4f}'
        )

Epoch [20/200], Loss: 0.6588
Epoch [40/200], Loss: 0.5559
Epoch [60/200], Loss: 0.4293
Epoch [80/200], Loss: 0.3615
Epoch [100/200], Loss: 0.3239
Epoch [120/200], Loss: 0.2957
Epoch [140/200], Loss: 0.2730
Epoch [160/200], Loss: 0.2523
Epoch [180/200], Loss: 0.2313
Epoch [200/200], Loss: 0.2106


In [118]:
with torch.no_grad():

    outputs = model(X_test)

    predicted = torch.argmax(
        outputs,
        dim=1
    )

In [119]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_ce.numpy(),
        predicted.numpy()
    )
)

              precision    recall  f1-score   support

           0       0.20      0.29      0.24        14
           1       0.89      0.84      0.86        99

    accuracy                           0.77       113
   macro avg       0.55      0.56      0.55       113
weighted avg       0.81      0.77      0.79       113



In [120]:
print(predicted.unique(return_counts=True))

(tensor([0, 1]), tensor([20, 93]))
